In [1]:
import torch
from torch import nn
from d2l import torch as d2l

In [2]:
# 2D Cross-Correlation

def corr2d(
    X: torch.Tensor, 
    K: torch.Tensor
) -> torch.Tensor:
    """Compute 2D cross-correlation"""
    
    input_height, input_width = X.shape
    kernel_height, kernel_width = K.shape
    
    output_height = input_height - kernel_height + 1
    output_width = input_width - kernel_width + 1
    
    Y = torch.zeros(
        (output_height, output_width),
        dtype=X.dtype,
        device=X.device,
    )
    
    for i in range(output_height):
        for j in range(output_width):
            window = X[
                i : i + kernel_height,
                j : j + kernel_width,
            ]
            Y[i, j] = (window * K).sum()
            
    return Y

In [3]:
X = torch.tensor([
    [0.0, 1.0, 2.0],
    [3.0, 4.0, 5.0],
    [6.0, 7.0, 8.0],
])

K = torch.tensor([
    [0.0, 1.0],
    [2.0, 3.0],
])

Y = corr2d(X, K)

print("X shape:", X.shape)
print("K shape:", K.shape)
print("Y shape:", Y.shape)
print(Y)

X shape: torch.Size([3, 3])
K shape: torch.Size([2, 2])
Y shape: torch.Size([2, 2])
tensor([[19., 25.],
        [37., 43.]])


In [4]:
# 2D Convolution Layer

class Conv2D(nn.Module):
    
    def __init__(
        self, 
        kernel_size: tuple[int, int],
    ) -> None:
        super().__init__()
        
        self.weight = nn.Parameter(
            torch.rand(kernel_size)
        )
        self.bias = nn.Parameter(
            torch.zeros(1)
        )
        
    def forward(
        self, 
        X: torch.Tensor,
    ) -> torch.Tensor:
        return corr2d(
            X,
            self.weight,
        ) + self.bias

In [5]:
# Edge Detection용 Image 생성

image = torch.ones((6, 8))
image[:, 2:6] = 0

print("image shape:", image.shape)
print(image)

image shape: torch.Size([6, 8])
tensor([[1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.]])


In [7]:
# Vertical Edge 검출과 Transpose

edge_kernel = torch.tensor([
    [1.0, -1.0]
])

edge_map = corr2d(
    image,
    edge_kernel,
)

transposed_edge_map = corr2d(
    image.T,
    edge_kernel,
)

print("edge kernel shape:", edge_kernel.shape)
print("edge map shape:", edge_map.shape)
print(edge_map)

print(
    "\ntransposed edge map shape:",
    transposed_edge_map.shape,
)
print(transposed_edge_map)

edge kernel shape: torch.Size([1, 2])
edge map shape: torch.Size([6, 7])
tensor([[ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.]])

transposed edge map shape: torch.Size([8, 5])
tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]])
